# ChaosOps AI — Colab training notebook

Minimum-viable GRPO training run for the **ChaosOps AI** multi-agent incident-response environment with rogue-agent detection.

**What this notebook does (top-to-bottom, ~2 hrs on a free T4):**
1. Installs deps (Unsloth 4-bit, TRL GRPO, transformers, peft)
2. Clones the project from GitHub (or pulls latest if already cloned)
3. Runs **scripted baselines** (random / heuristic / oracle) → produces `baseline_curve.png`
4. Runs a **GRPO training pass** on Qwen 2.5 0.5B — 800 gradient steps
5. Plots the training metrics (raw + smoothed)

> T4-friendly settings: Qwen 2.5 **0.5B** Instruct, 4-bit quant, LoRA r=16, **800 gradient steps**, **group size 4**, max_seq_length=1024. The real hackathon training run uses **Qwen 2.5 7B** + LoRA r=32 on onsite HF-credit GPUs.

**Pitch materials produced:**
- `artifacts/baseline/baseline_curve.png` — Random vs. Heuristic vs. Oracle reward by tier
- `artifacts/chaosops-grpo/training_metrics.json` + rendered learning curve
- LoRA checkpoint in `artifacts/chaosops-grpo/lora_adapter/`

In [ ]:
# 1. Verify GPU
!nvidia-smi | head -n 20

## 1 · Install dependencies

Unsloth handles the 4-bit loader + LoRA attach; TRL provides GRPOTrainer.

In [ ]:
%%capture
!pip install --upgrade --quiet \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    "trl>=0.11.0" \
    "transformers>=4.44" \
    "peft>=0.12" \
    "accelerate>=0.33" \
    "bitsandbytes>=0.43" \
    "datasets" \
    "rich" \
    "pydantic>=2" \
    "matplotlib"

## 2 · Clone ChaosOps AI

In [ ]:
%cd /content
import os
if os.path.isdir('/content/chaos_ops/.git'):
    !cd /content/chaos_ops && git pull --ff-only
else:
    !rm -rf /content/chaos_ops
    !git clone https://github.com/vatsalllll/chaos_ops.git /content/chaos_ops
%cd /content/chaos_ops
!ls

In [ ]:
# Install ChaosOps in editable mode so `python -m chaosops.*` works from any cwd
!pip install -e /content/chaos_ops --quiet
import chaosops
print("chaosops package loaded from:", chaosops.__file__)

## 3 · Scripted baselines (the "before training" numbers)

This is the numeric evidence for the **Reward Improvement** rubric criterion. Runs in <2 min on CPU — no GPU needed.

In [ ]:
!python -m chaosops.train.baseline --episodes-per-type 5

In [ ]:
from IPython.display import Image
Image("artifacts/baseline/baseline_curve.png")

## 4 · GRPO training pass (Qwen 2.5 0.5B, T4-sized)

**800 optimisation steps** × group size **4** on the 0.5B model — enough gradient updates to see a clear rising reward curve. The pipeline is identical to the onsite 7B run; only the knobs are smaller.

Expect **~2 hours** of training after the model download completes. Keep the Colab tab in the foreground to avoid idle disconnects. If you hit OOM with `--group-size 4`, drop it to `2` — the trend still shows, just noisier.

In [ ]:
!python -m chaosops.train.grpo_train \
    --model-name Qwen/Qwen2.5-0.5B-Instruct \
    --total-episodes 800 \
    --group-size 4 \
    --team-weight 0.6 \
    --log-every 10 \
    --max-seq-length 1024 \
    --lora-rank 16 \
    --output-dir artifacts/chaosops-grpo \
    --start-tier easy

## 5 · Plot the learning curve

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

metrics_path = Path('artifacts/chaosops-grpo/training_metrics.json')
metrics = json.loads(metrics_path.read_text())

episodes = [m['episode'] for m in metrics]
combined = [m.get('mean_combined_reward', float('nan')) for m in metrics]

def moving_average(xs, window):
    if len(xs) < window:
        return xs
    out = []
    for i in range(len(xs)):
        lo = max(0, i - window + 1)
        out.append(sum(xs[lo:i+1]) / (i - lo + 1))
    return out

window = max(3, len(combined) // 10)
smoothed = moving_average(combined, window)

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=150)
ax.plot(episodes, combined, label='reward (raw)', color='#95a5a6', linewidth=1.2, alpha=0.7)
ax.plot(episodes, smoothed, label=f'reward (moving avg, window={window})', color='#27ae60', linewidth=2.5)
ax.axhline(0, color='#888', linewidth=0.6)
ax.set_xlabel('gradient step')
ax.set_ylabel('mean reward over GRPO group')
ax.set_title('ChaosOps AI — GRPO learning curve (Qwen 2.5 0.5B, Colab T4)')
ax.legend(loc='lower right')
ax.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()
plt.savefig('artifacts/chaosops-grpo/learning_curve.png')
plt.show()

print(f"logged points: {len(metrics)}")
print(f"first reward:  {combined[0]:+.2f}")
print(f"last reward:   {combined[-1]:+.2f}")
print(f"delta:         {combined[-1] - combined[0]:+.2f}")


## 6 · Next steps (onsite, HF-credit GPU)

Once you get bigger GPUs, bump the knobs:

```bash
python -m chaosops.train.grpo_train \
    --model-name Qwen/Qwen2.5-7B-Instruct \
    --total-episodes 800 \
    --group-size 4 \
    --team-weight 0.6 \
    --log-every 10 \
    --max-seq-length 1536 \
    --lora-rank 32 \
    --output-dir artifacts/chaosops-grpo-7b \
    --start-tier easy
```

Expected outcome for the pitch slide:
- MTTR on HARD: **~10 steps → ~4 steps**
- Rogue-catch rate: **~20% → >70%**
- Mean HARD reward: **−237 → >+50**